# Nonlinear Hawkes: Mean-Field Expansion

**Action:**
$$S = \sum_{i} \left[ \tilde{n}_i^\top \dot{n}_i + (e^{\tilde{n}_i}-1)^\top \phi(v_i) + \tilde{v}_i^\top\left((\tau\partial_t+1)v_i - E_i - \sum_j w_{ij}\, g\ast\dot{n}_j\right) \right]$$

**Fluctuation expansion** (physical fields only):
$$n_i = n_i^* + \delta n_i, \qquad v_i = v_i^* + \delta v_i$$

Response fields $\tilde{n}_i$, $\tilde{v}_i$ are **not** expanded — they are integration variables.

**Mean-field equations** (cancel tadpole terms):
$$\phi(v_i^*) = n_i^*, \qquad v_i^* = E_i + \sum_j w_{ij} n_j^*$$

After substitution and MF cancellation we classify each term by **two independent degrees**:
- $n_{\tilde{}} =$ degree in response fields $\{\tilde{n}_i, \tilde{v}_i\}$
- $n_{\rm phys} =$ degree in physical fluctuations $\{\delta n_i, \delta v_i, \partial_t\delta n_i, \ldots\}$

| $(n_{\tilde{}},\, n_{\rm phys})$ | Name | Example |
|---|---|---|
| $(1,\,1)$ | **Free action** $S_0$ | $\tilde{n}_i\,\partial_t\delta n_i$ |
| $(2,\,0)$ | **Noise kernel** | $\tilde{n}_i^2\, n_i^*$ |
| $n_{\tilde{}}+n_{\rm phys}\geq 3$ | **Vertices** $S_{\rm int}$ | $\tilde{n}_i\,(\delta v_i)^2$, $\tilde{n}_i^2\,\delta v_i$, … |

In [ ]:
import sympy as sp
from sympy import symbols, Function, exp, factorial, Add, Mul, Rational
from IPython.display import display, Math

sp.init_printing(use_latex='mathjax')

# ---------------------------------------------------------------------------
# Conv and IP — explicit operator notation
#
#   Conv(kappa, f)  represents  (κ ∗ f)(t) = ∫ κ(t-s) f(s) ds
#   IP(a, b)        represents  a^⊤ b      = ∫ a(t) b(t) dt
#
# Every bilinear term in the free action is written as  IP(response, Conv(kernel, field)).
# This makes it explicit that:
#   - even a scalar coefficient c really means Conv(c·δ, field)
#   - g is a convolution kernel, not a multiplicative scalar
#   - (τ∂_t+1) corresponds to the kernel τδ'(t)+δ(t)
# ---------------------------------------------------------------------------

class Conv(sp.Function):
    """Conv(kappa, f) — convolution (κ∗f)(t) = ∫κ(t−s)f(s)ds"""
    @classmethod
    def eval(cls, kappa, f): return None

    def _latex(self, printer):
        k, f = self.args
        return (r'\left(' + printer.doprint(k)
                + r' \ast ' + printer.doprint(f) + r'\right)')

    def _sympystr(self, printer):
        k, f = self.args
        return f'Conv({printer.doprint(k)}, {printer.doprint(f)})'


class IP(sp.Function):
    """IP(a, b) — inner product a^⊤b = ∫a(t)b(t)dt"""
    @classmethod
    def eval(cls, a, b): return None

    def _latex(self, printer):
        a, b = self.args
        return printer.doprint(a) + r'^{\top} ' + printer.doprint(b)

    def _sympystr(self, printer):
        a, b = self.args
        return f'IP({printer.doprint(a)}, {printer.doprint(b)})'

## 1. Symbol definitions

In [ ]:
N_pop = 2
pop   = list(range(N_pop))

# ---------------------------------------------------------------------------
# MF background
# ---------------------------------------------------------------------------
nstar = [symbols(f'nstar{i+1}', positive=True) for i in pop]
vstar = [symbols(f'vstar{i+1}')                for i in pop]
E     = [symbols(f'E{i+1}')                    for i in pop]
tau   = symbols('tau', positive=True)

w = [[symbols(f'w{i+1}{j+1}') for j in pop] for i in pop]

# ---------------------------------------------------------------------------
# Physical fluctuation fields  (expanded around MF background)
#   dn[i] = δṅ_i  — fluctuation in the spike train field
#   dv[i] = δv_i  — fluctuation in the voltage field
# ---------------------------------------------------------------------------
dn = [symbols(f'dn{i+1}') for i in pop]
dv = [symbols(f'dv{i+1}') for i in pop]

# ---------------------------------------------------------------------------
# Response fields  (NOT expanded; full fields, not fluctuations)
#   nt[i] = ñ_i   — response field conjugate to spike train
#   vt[i] = ṽ_i   — response field conjugate to voltage
# ---------------------------------------------------------------------------
nt = [symbols(f'nt{i+1}') for i in pop]
vt = [symbols(f'vt{i+1}') for i in pop]

# ---------------------------------------------------------------------------
# Convolution kernel symbols
#   delta_D  = δ(t)   Dirac delta         Conv(δ,  f) = f
#   delta_Dp = δ'(t)  derivative of delta  Conv(δ', f) = ∂_t f
#   gop      = g(t)   synaptic filter      Conv(g,  f) = g∗f
# ---------------------------------------------------------------------------
delta_D  = symbols(r'\delta')
delta_Dp = symbols(r"\delta'")
gop      = symbols('g')
Dt       = symbols('Dt')   # algebraic stand-in for ∂_t (sector counting only)

# ---------------------------------------------------------------------------
# Taylor coefficients of phi at MF background
# ---------------------------------------------------------------------------
phi0s = [symbols(f'phi0_{i+1}') for i in pop]
phi1s = [symbols(f'phi1_{i+1}') for i in pop]
phi2s = [symbols(f'phi2_{i+1}') for i in pop]
phi3s = [symbols(f'phi3_{i+1}') for i in pop]

# ---------------------------------------------------------------------------
# Degree-counting sets
#   TILDE_SYMS : response fields  (n_tilde count)
#   PHYS_SYMS  : physical fluctuation fields  (n_phys count)
# ---------------------------------------------------------------------------
TILDE_SYMS = nt + vt
PHYS_SYMS  = dn + dv

TAYLOR_ORDER = 4

print('Physical fluctuation fields:', PHYS_SYMS)
print('Response fields:            ', TILDE_SYMS)
print('Kernel symbols:             ', [delta_D, delta_Dp, gop])

## 2. Degree counting

Each term in the expanded action is classified by two independent degrees:
- `n_tilde`: degree in response fields `dnt`, `dvt`
- `n_phys`: degree in physical fluctuation fields `dn`, `dv`, `Dtdn`, `gDtdn`, `TauDtdv`

The free action is exactly the `(n_tilde=1, n_phys=1)` sector.  
The noise kernel is `(n_tilde=2, n_phys=0)`, `(n_tilde=3, n_phys=0)`, etc.  
Vertices have total degree ≥ 3 (any combination summing to ≥ 3).

In [ ]:
def sym_degree(term, syms):
    """Total degree of a monomial `term` in the list of symbols `syms`."""
    return sum(sp.degree(term, x) for x in syms)


def collect_by_tilde_phys(expr, tilde_syms, phys_syms):
    """
    Split `expr` into a dict {(n_tilde, n_phys): sub_expression}.
    n_tilde = degree in tilde_syms, n_phys = degree in phys_syms.
    """
    expr = sp.expand(expr)
    result = {}
    for term in Add.make_args(expr):
        n_t = sym_degree(term, tilde_syms)
        n_p = sym_degree(term, phys_syms)
        key = (n_t, n_p)
        result[key] = result.get(key, sp.Integer(0)) + term
    return result


def pretty_print_sectors(by_tp, label=''):
    if label:
        print(label)
    for (n_t, n_p), expr in sorted(by_tp.items()):
        e = sp.expand(expr)
        if e == 0:
            continue
        print(f'  (n_tilde={n_t}, n_phys={n_p}):')
        display(e)


print('Degree-counting helpers defined.')
print('Key sectors:')
print('  (1,1) -> free action')
print('  (2,0) -> noise kernel')
print('  total >= 3 -> vertices')

## 3. Taylor expansions of the nonlinear functions

Expand in the respective fluctuation variable (NOT in a small parameter):
$$e^{\tilde{n}_i} - 1 = \tilde{n}_i + \tfrac{1}{2}\tilde{n}_i^2 + \tfrac{1}{6}\tilde{n}_i^3 + \cdots$$
$$\phi(v_i^* + \delta v_i) = \phi_0^{(i)} + \phi_1^{(i)}\,\delta v_i + \tfrac{1}{2}\phi_2^{(i)}\,(\delta v_i)^2 + \cdots$$

In [ ]:
def exp_m1_Taylor(nt_i, order=TAYLOR_ORDER):
    """Taylor expansion of (exp(nt_i) - 1) in nt_i to given order."""
    return sum(nt_i**k / factorial(k) for k in range(1, order + 1))


def phi_Taylor(i, dv_i, order=TAYLOR_ORDER):
    """Taylor expansion of phi(v_i* + dv_i) in dv_i to given order."""
    derivs = [phi0s[i], phi1s[i], phi2s[i], phi3s[i]]
    return sum(derivs[k] * dv_i**k / factorial(k)
               for k in range(min(order + 1, len(derivs))))


print('exp(nt1) - 1 =')
display(exp_m1_Taylor(nt[0]))

print('phi(v1* + dv1) =')
display(phi_Taylor(0, dv[0]))

## 4. Assemble the action

Substitute $\dot{n}_i = n_i^* + \delta\dot{n}_i$ and $v_i = v_i^* + \delta v_i$, where $\dot{n}_i$ is the spike train field.

| Action term | Expansion | Symbolic |
|---|---|---|
| $\tilde{n}_i^\top \dot{n}_i$ | $\tilde{n}_i(n_i^* + \delta\dot{n}_i)$ | `dnt[i]*(nstar[i] + dr[i])` |
| $g\ast\dot{n}_j$ (background) | $g\ast n_j^* = n_j^*$ | `nstar[j]` (cancels via MF) |
| $g\ast\delta\dot{n}_j$ (fluctuation) | filter on spike train fluctuation | `gop * dr[j]` — **no** `Dt` |
| $(\tau\partial_t+1)\delta v_i$ | explicit $\partial_t$ from voltage eq. | `(tau*Dt + 1) * dv[i]` |

`Dt` ($\partial_t$) appears **only** in the voltage operator — the user explicitly wrote $\partial_t$ there. The coupling $g\ast\dot{n}_j$ has no extra derivative because $\dot{n}$ is already the fundamental field.

**MF saddle-point condition** $\delta S/\delta\tilde{n}_i\big|_\text{bg}=0$:
$$\dot{n}_i^* + \phi(v_i^*) = 0 \implies \dot{n}_i^* = -n_i^*$$

In [ ]:
ndot_bg = [-nstar[i] for i in pop]   # MF saddle: ṅ_i* = -phi(v_i*) = -n_i*


def build_Si(i, order=TAYLOR_ORDER):
    # S1: ñ_i * ṅ_i,  ṅ_i = ṅ_i* + δṅ_i
    S1 = nt[i] * (ndot_bg[i] + dn[i])

    # S2: (exp(ñ_i) - 1) * phi(v_i* + δv_i)
    S2 = sp.expand(exp_m1_Taylor(nt[i], order) * phi_Taylor(i, dv[i], order))

    # S3: ṽ_i * [(τ∂_t+1)(v_i*+δv_i) - E_i - Σ_j w_ij g∗ṅ_j]
    volt_bg   = vstar[i]
    volt_fl   = (tau * Dt + 1) * dv[i]
    filter_bg = sum(w[i][j] * nstar[j]    for j in pop)
    filter_fl = sum(w[i][j] * gop * dn[j] for j in pop)

    bracket = (volt_bg + volt_fl) - E[i] - (filter_bg + filter_fl)
    S3 = vt[i] * sp.expand(bracket)

    return sp.expand(S1 + S2 + S3)


S_pieces_raw = [build_Si(i) for i in pop]
S_raw = sp.expand(sum(S_pieces_raw))

print('=== Raw action (before MF substitution) ===')
raw_by_tp = collect_by_tilde_phys(S_raw, TILDE_SYMS, PHYS_SYMS)
pretty_print_sectors(raw_by_tp)

print()
print('--- (1,0) sector before MF [should cancel after phi0s → nstar]:')
display(sp.expand(raw_by_tp.get((1, 0), sp.Integer(0))))

## 5. Apply mean-field equations

**MF equations:**
$$\phi(v_i^*) = n_i^* \implies \phi_0^{(i)} = n_i^*$$
$$v_i^* - E_i - \sum_j w_{ij} n_j^* = 0$$

The second equation cancels the degree-1 bracket in $S_3$.  
The first equation cancels the degree-1 term $\tilde{n}_i \phi_0^{(i)}$ in $S_2$.

In [ ]:
# ---------------------------------------------------------------------------
# MF substitutions
#   1. phi(v_i*) = n_i*          [Poisson saddle + voltage eq. combined]
#   2. v_i* = E_i + Σ_j w_ij n_j*   [voltage MF eq. → cancels S3 background bracket]
# ---------------------------------------------------------------------------
mf_phi  = {phi0s[i]: nstar[i]
           for i in pop}
mf_volt = {vstar[i]: E[i] + sum(w[i][j] * nstar[j] for j in pop)
           for i in pop}

S_mf = sp.expand(S_raw.subs(mf_phi).subs(mf_volt))

print('=== Action after MF substitution ===')
mf_by_tp = collect_by_tilde_phys(S_mf, TILDE_SYMS, PHYS_SYMS)
pretty_print_sectors(mf_by_tp)

## 6. Free action and interaction vertices

In [ ]:
def assert_zero_sector(by_tp, key, label):
    val = sp.expand(by_tp.get(key, sp.Integer(0)))
    ok  = val == 0
    print(f'  [{"PASS" if ok else "FAIL"}]  (n_tilde={key[0]}, n_phys={key[1]})  {label}')
    if not ok:
        display(val)
    return ok

print('=== Sanity checks ===')
all_pass = True
all_pass &= assert_zero_sector(mf_by_tp, (0,0), 'constant')
all_pass &= assert_zero_sector(mf_by_tp, (1,0), 'linear in ñ/ṽ only — Poisson tadpole')
all_pass &= assert_zero_sector(mf_by_tp, (0,1), 'linear in physical only — should never appear')
print()
print('All checks passed.' if all_pass else 'Some checks FAILED.')

In [ ]:
S_free = sp.expand(mf_by_tp.get((1, 1), sp.Integer(0)))

print('=== Free action S_free  [(n_tilde=1, n_phys=1)] ===')
display(S_free)
print()
print('LaTeX:')
print(sp.latex(S_free))
print()

# Structural check: every term must be exactly (n_tilde=1, n_phys=1)
print('--- Structural check ---')
fail = False
for term in Add.make_args(sp.expand(S_free)):
    n_t = sym_degree(term, TILDE_SYMS)
    n_p = sym_degree(term, PHYS_SYMS)
    if (n_t, n_p) != (1, 1):
        print(f'  [FAIL] unexpected (n_tilde={n_t}, n_phys={n_p}): {term}')
        fail = True
if not fail:
    print('  [PASS] all terms have exactly (n_tilde=1, n_phys=1)')

In [ ]:
print('=== Noise/source kernel sectors  [(n_tilde>=2, n_phys=0)] ===')
found = False
for (n_t, n_p), expr in sorted(mf_by_tp.items()):
    if n_t >= 2 and n_p == 0:
        e = sp.expand(expr)
        if e != 0:
            found = True
            print(f'  (n_tilde={n_t}, n_phys={n_p}):')
            display(e)
if not found:
    print('  (none)')

In [ ]:
print('=== Interaction vertices  [n_tilde + n_phys >= 3] ===')
found = False
for (n_t, n_p), expr in sorted(mf_by_tp.items()):
    if n_t + n_p >= 3:
        e = sp.expand(expr)
        if e != 0:
            found = True
            print(f'  (n_tilde={n_t}, n_phys={n_p}):')
            display(e)
            print('  LaTeX:', sp.latex(e))
            print()
if not found:
    print(f'  (none up to Taylor order {TAYLOR_ORDER})')

## 7. Extract bilinear structure of the free action

Read off the coefficient of each response-field × physical-field pair in $S_{\rm free}$.  
This gives the inverse propagator matrix entries.

In [ ]:
S_free_conv = sp.Integer(0)

for i in pop:
    S_free_conv += IP(nt[i], Conv(delta_D, dn[i]))
    S_free_conv += IP(nt[i], Conv(phi1s[i] * delta_D, dv[i]))
    S_free_conv += IP(vt[i], Conv(tau * delta_Dp + delta_D, dv[i]))
    for j in pop:
        S_free_conv -= IP(vt[i], Conv(w[i][j] * gop, dn[j]))

print('=== Free action S_free in Conv/IP notation ===')
display(S_free_conv)
print()
print('LaTeX:')
print(sp.latex(S_free_conv))

## 8. Symbol legend

### Fields
| Symbol | Meaning |
|--------|---------|
| `dn[i]` | $\delta\dot{n}_i$ — spike train fluctuation |
| `dv[i]` | $\delta v_i$ — voltage fluctuation |
| `nt[i]` | $\tilde{n}_i$ — response field (full field, not a fluctuation) |
| `vt[i]` | $\tilde{v}_i$ — response field (full field, not a fluctuation) |
| `nstar[i]` | $n_i^* = \phi(v_i^*)$ — MF rate |

### Convolution kernels
| Symbol | Kernel $\kappa(t)$ | `Conv(κ, f)` computes |
|--------|-------------------|----------------------|
| `delta_D` | $\delta(t)$ | $f(t)$ — identity |
| `delta_Dp` | $\delta'(t)$ | $\partial_t f(t)$ |
| `gop` | $g(t)$ | $(g\ast f)(t)$ |
| `phi1s[i]*delta_D` | $\phi'(v_i^*)\,\delta(t)$ | $\phi'(v_i^*)\,f(t)$ |
| `tau*delta_Dp + delta_D` | $\tau\delta'(t)+\delta(t)$ | $(\tau\partial_t+1)f(t)$ |
| `w[i][j]*gop` | $w_{ij}\,g(t)$ | $w_{ij}(g\ast f)(t)$ |

### Sector classification
| $(n_{\tilde{}},\, n_{\rm phys})$ | Name |
|---|---|
| $(1,1)$ | Free action (inverse propagator) |
| $(\geq2,\,0)$ | Noise kernel (Poisson cumulants) |
| total $\geq3$ | Interaction vertices |